In [1]:
"""
Synthetic mule-network data generator
------------------------------------

Generates:
1. accounts.csv              -> account-level metadata
2. transactions.csv          -> transaction-level data
3. account_labels.csv        -> hidden truth labels for accounts
4. mule_edges_groundtruth.csv-> edges that belong to the planted mule network

Use cases:
- graph anomaly detection
- mule ring detection
- AML feature engineering
- link prediction / community detection
- supervised node classification
- transaction monitoring experiments

This is a plausible synthetic simulator for fintech-style AML data.
It is NOT a reproduction of any real company's internal system.

Requirements:
    pip install pandas numpy

Optional:
    pip install networkx
"""

from __future__ import annotations

import math
import random
import uuid
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd


# ----------------------------
# Configuration
# ----------------------------

SEED = 42
N_ACCOUNTS = 2500
N_NORMAL_TXNS = 22000
N_MULE_CLUSTERS = 8

# cluster sizes
MULE_CLUSTER_MIN = 8
MULE_CLUSTER_MAX = 22

# countries
COUNTRIES = [
    "EE", "LV", "LT", "DE", "NL", "BE", "FR", "ES", "IT", "PL", "RO", "CZ", "GB"
]

HIGH_RISK_COUNTRIES = {"RO"}

CURRENCIES = {
    "EE": "EUR",
    "LV": "EUR",
    "LT": "EUR",
    "DE": "EUR",
    "NL": "EUR",
    "BE": "EUR",
    "FR": "EUR",
    "ES": "EUR",
    "IT": "EUR",
    "PL": "PLN",
    "RO": "RON",
    "CZ": "CZK",
    "GB": "GBP",
}

BUSINESS_HOURS_START = 8
BUSINESS_HOURS_END = 20

random.seed(SEED)
np.random.seed(SEED)


# ----------------------------
# Data classes
# ----------------------------

@dataclass
class Account:
    account_id: str
    country: str
    created_at: datetime
    age_days: int
    is_business: int
    kyc_risk: float
    device_id: str
    ip_id: str
    balance_seed: float
    segment: str  # "normal" / "mule"
    mule_role: str  # "none" / recruiter / collector / disperser
    cluster_id: int  # -1 for non-mule


# ----------------------------
# Helper functions
# ----------------------------

def rand_datetime(start: datetime, end: datetime) -> datetime:
    delta = end - start
    seconds = int(delta.total_seconds())
    return start + timedelta(seconds=random.randint(0, seconds))


def random_id(prefix: str) -> str:
    return f"{prefix}_{uuid.uuid4().hex[:12]}"


def clipped_normal(mean: float, sd: float, low: float, high: float) -> float:
    x = np.random.normal(mean, sd)
    return float(np.clip(x, low, high))


def sample_amount_normal() -> float:
    """
    Normal users: mostly small/medium retail payments.
    Lognormal gives a heavy-tailed but sane transaction size distribution.
    """
    amt = np.random.lognormal(mean=3.6, sigma=0.8)  # around tens to low hundreds
    return round(float(min(amt, 4000.0)), 2)


def sample_amount_mule(layer: str) -> float:
    """
    Mule patterns:
    - incoming amounts often clustered just under review thresholds
    - outgoing amounts split
    """
    if layer == "incoming":
        base = np.random.choice([85, 120, 180, 240, 480, 950], p=[0.18, 0.18, 0.18, 0.16, 0.16, 0.14])
        amt = np.random.normal(base, base * 0.08)
    else:
        base = np.random.choice([75, 95, 110, 140, 220, 470], p=[0.18, 0.18, 0.18, 0.18, 0.16, 0.12])
        amt = np.random.normal(base, base * 0.07)
    return round(float(max(8.0, amt)), 2)


def sample_business_hour_timestamp(day_start: datetime) -> datetime:
    hour = random.randint(BUSINESS_HOURS_START, BUSINESS_HOURS_END)
    minute = random.randint(0, 59)
    second = random.randint(0, 59)
    return day_start.replace(hour=hour, minute=minute, second=second)


def sample_offhour_timestamp(day_start: datetime) -> datetime:
    hour = random.choice(list(range(0, BUSINESS_HOURS_START)) + list(range(BUSINESS_HOURS_END + 1, 24)))
    minute = random.randint(0, 59)
    second = random.randint(0, 59)
    return day_start.replace(hour=hour, minute=minute, second=second)


# ----------------------------
# Account generation
# ----------------------------

def generate_accounts(
    n_accounts: int,
    n_mule_clusters: int,
    start_date: datetime,
    end_date: datetime,
) -> Tuple[pd.DataFrame, Dict[str, Account], List[List[str]]]:
    accounts: Dict[str, Account] = {}
    mule_clusters: List[List[str]] = []

    # shared device/ip pools for suspicious re-use
    suspicious_device_pool = [random_id("dev_shared") for _ in range(max(20, n_mule_clusters * 3))]
    suspicious_ip_pool = [random_id("ip_shared") for _ in range(max(20, n_mule_clusters * 3))]

    # 1) create planted mule clusters
    mule_account_ids = set()

    for cluster_id in range(n_mule_clusters):
        size = random.randint(MULE_CLUSTER_MIN, MULE_CLUSTER_MAX)
        cluster_accounts = []

        # structure: a few collectors, many mules, a few dispersers
        n_collectors = max(1, size // 6)
        n_dispersers = max(1, size // 8)
        n_recruiters = 1

        roles = (
            ["recruiter"] * n_recruiters
            + ["collector"] * n_collectors
            + ["disperser"] * n_dispersers
        )
        while len(roles) < size:
            roles.append("mule")

        random.shuffle(roles)

        shared_country = random.choice(["EE", "LV", "LT", "DE", "NL"])
        shared_device = random.choice(suspicious_device_pool)
        shared_ip = random.choice(suspicious_ip_pool)

        for role in roles:
            account_id = random_id("acct")
            created_at = rand_datetime(start_date, end_date - timedelta(days=7))
            age_days = max(1, (end_date - created_at).days)

            # mules tend to be newer, lower-quality KYC, more shared infrastructure
            country = shared_country if random.random() < 0.75 else random.choice(COUNTRIES)
            device_id = shared_device if random.random() < 0.55 else random_id("dev")
            ip_id = shared_ip if random.random() < 0.45 else random_id("ip")

            acc = Account(
                account_id=account_id,
                country=country,
                created_at=created_at,
                age_days=age_days,
                is_business=int(random.random() < 0.08),
                kyc_risk=clipped_normal(0.72, 0.12, 0.35, 0.99),
                device_id=device_id,
                ip_id=ip_id,
                balance_seed=round(np.random.lognormal(mean=4.0, sigma=0.7), 2),
                segment="mule",
                mule_role=role,
                cluster_id=cluster_id,
            )
            accounts[account_id] = acc
            cluster_accounts.append(account_id)
            mule_account_ids.add(account_id)

        mule_clusters.append(cluster_accounts)

    # 2) create normal accounts
    while len(accounts) < n_accounts:
        account_id = random_id("acct")
        created_at = rand_datetime(start_date - timedelta(days=365), end_date - timedelta(days=5))
        age_days = max(1, (end_date - created_at).days)

        country = random.choice(COUNTRIES)
        is_business = int(random.random() < 0.18)

        acc = Account(
            account_id=account_id,
            country=country,
            created_at=created_at,
            age_days=age_days,
            is_business=is_business,
            kyc_risk=clipped_normal(0.23, 0.15, 0.01, 0.9),
            device_id=random_id("dev"),
            ip_id=random_id("ip"),
            balance_seed=round(np.random.lognormal(mean=4.7, sigma=0.9), 2),
            segment="normal",
            mule_role="none",
            cluster_id=-1,
        )
        accounts[account_id] = acc

    accounts_df = pd.DataFrame([
        {
            "account_id": a.account_id,
            "country": a.country,
            "created_at": a.created_at,
            "age_days": a.age_days,
            "is_business": a.is_business,
            "kyc_risk": a.kyc_risk,
            "device_id": a.device_id,
            "ip_id": a.ip_id,
            "balance_seed": a.balance_seed,
        }
        for a in accounts.values()
    ])

    return accounts_df, accounts, mule_clusters


# ----------------------------
# Transaction generation
# ----------------------------

def generate_normal_transactions(
    accounts: Dict[str, Account],
    n_txns: int,
    start_date: datetime,
    end_date: datetime,
) -> pd.DataFrame:
    normal_accounts = [a for a in accounts.values() if a.segment == "normal"]
    all_accounts = list(accounts.values())

    rows = []
    for _ in range(n_txns):
        src = random.choice(normal_accounts)

        # mostly normal-to-normal, occasionally to businesses or random accounts
        if random.random() < 0.92:
            dst = random.choice(normal_accounts)
        else:
            dst = random.choice(all_accounts)

        while dst.account_id == src.account_id:
            dst = random.choice(all_accounts)

        ts = rand_datetime(start_date, end_date)
        amt = sample_amount_normal()

        rows.append({
            "txn_id": random_id("txn"),
            "timestamp": ts,
            "src_account": src.account_id,
            "dst_account": dst.account_id,
            "amount": amt,
            "src_country": src.country,
            "dst_country": dst.country,
            "currency": CURRENCIES.get(src.country, "EUR"),
            "channel": random.choices(
                ["bank_transfer", "card_topup", "p2p", "merchant"],
                weights=[0.45, 0.15, 0.30, 0.10],
                k=1
            )[0],
            "is_cross_border": int(src.country != dst.country),
            "is_off_hours": int(ts.hour < BUSINESS_HOURS_START or ts.hour > BUSINESS_HOURS_END),
            "is_planted_mule_edge": 0,
            "planted_cluster_id": -1,
        })

    return pd.DataFrame(rows)


def generate_mule_transactions(
    accounts: Dict[str, Account],
    mule_clusters: List[List[str]],
    start_date: datetime,
    end_date: datetime,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    planted_edges = []

    normal_accounts = [a for a in accounts.values() if a.segment == "normal"]

    for cluster_id, cluster in enumerate(mule_clusters):
        cluster_objs = [accounts[aid] for aid in cluster]
        recruiters = [a for a in cluster_objs if a.mule_role == "recruiter"]
        collectors = [a for a in cluster_objs if a.mule_role == "collector"]
        dispersers = [a for a in cluster_objs if a.mule_role == "disperser"]
        mules = [a for a in cluster_objs if a.mule_role == "mule"]

        if not collectors:
            collectors = mules[:1]
        if not dispersers:
            dispersers = mules[-1:]

        # cluster active window: bursty short campaign
        campaign_start = rand_datetime(start_date + timedelta(days=15), end_date - timedelta(days=20))
        active_days = random.randint(2, 8)

        for day_offset in range(active_days):
            day = campaign_start + timedelta(days=day_offset)

            # Step 1: external/normal sources feed collectors and mules
            n_incoming = random.randint(15, 45)
            recipients = collectors + mules

            for _ in range(n_incoming):
                src = random.choice(normal_accounts)
                dst = random.choice(recipients)

                # mules often get repeated near-threshold deposits and off-hour activity
                ts = sample_offhour_timestamp(day) if random.random() < 0.52 else sample_business_hour_timestamp(day)
                amt = sample_amount_mule("incoming")

                rows.append({
                    "txn_id": random_id("txn"),
                    "timestamp": ts,
                    "src_account": src.account_id,
                    "dst_account": dst.account_id,
                    "amount": amt,
                    "src_country": src.country,
                    "dst_country": dst.country,
                    "currency": CURRENCIES.get(src.country, "EUR"),
                    "channel": random.choices(
                        ["bank_transfer", "p2p", "card_topup"],
                        weights=[0.35, 0.55, 0.10],
                        k=1
                    )[0],
                    "is_cross_border": int(src.country != dst.country),
                    "is_off_hours": int(ts.hour < BUSINESS_HOURS_START or ts.hour > BUSINESS_HOURS_END),
                    "is_planted_mule_edge": 1,
                    "planted_cluster_id": cluster_id,
                })
                planted_edges.append((src.account_id, dst.account_id, cluster_id, "inbound"))

            # Step 2: mules forward to collectors (fan-in)
            n_fanin = random.randint(12, 35)
            fanin_sources = mules if mules else recipients
            for _ in range(n_fanin):
                src = random.choice(fanin_sources)
                dst = random.choice(collectors)
                if src.account_id == dst.account_id:
                    continue

                ts = sample_offhour_timestamp(day) if random.random() < 0.7 else sample_business_hour_timestamp(day)
                amt = sample_amount_mule("outgoing")

                rows.append({
                    "txn_id": random_id("txn"),
                    "timestamp": ts,
                    "src_account": src.account_id,
                    "dst_account": dst.account_id,
                    "amount": amt,
                    "src_country": src.country,
                    "dst_country": dst.country,
                    "currency": CURRENCIES.get(src.country, "EUR"),
                    "channel": random.choices(
                        ["p2p", "bank_transfer"],
                        weights=[0.7, 0.3],
                        k=1
                    )[0],
                    "is_cross_border": int(src.country != dst.country),
                    "is_off_hours": int(ts.hour < BUSINESS_HOURS_START or ts.hour > BUSINESS_HOURS_END),
                    "is_planted_mule_edge": 1,
                    "planted_cluster_id": cluster_id,
                })
                planted_edges.append((src.account_id, dst.account_id, cluster_id, "fanin"))

            # Step 3: collectors split onward to dispersers / exits (fan-out)
            n_fanout = random.randint(10, 28)
            for _ in range(n_fanout):
                src = random.choice(collectors)

                # sometimes send to internal dispersers, sometimes to random external-ish accounts
                if random.random() < 0.72:
                    dst = random.choice(dispersers)
                else:
                    dst = random.choice(normal_accounts)

                if src.account_id == dst.account_id:
                    continue

                ts = sample_offhour_timestamp(day) if random.random() < 0.58 else sample_business_hour_timestamp(day)
                amt = round(sample_amount_mule("outgoing") * np.random.uniform(1.0, 2.5), 2)

                rows.append({
                    "txn_id": random_id("txn"),
                    "timestamp": ts,
                    "src_account": src.account_id,
                    "dst_account": dst.account_id,
                    "amount": amt,
                    "src_country": src.country,
                    "dst_country": dst.country,
                    "currency": CURRENCIES.get(src.country, "EUR"),
                    "channel": random.choices(
                        ["bank_transfer", "p2p"],
                        weights=[0.65, 0.35],
                        k=1
                    )[0],
                    "is_cross_border": int(src.country != dst.country),
                    "is_off_hours": int(ts.hour < BUSINESS_HOURS_START or ts.hour > BUSINESS_HOURS_END),
                    "is_planted_mule_edge": 1,
                    "planted_cluster_id": cluster_id,
                })
                planted_edges.append((src.account_id, dst.account_id, cluster_id, "fanout"))

            # Step 4: occasional chain hops between mules to obscure path
            n_chain = random.randint(4, 12)
            chain_nodes = collectors + mules + dispersers
            if len(chain_nodes) >= 2:
                for _ in range(n_chain):
                    src, dst = random.sample(chain_nodes, 2)
                    ts = sample_offhour_timestamp(day)
                    amt = sample_amount_mule("outgoing")

                    rows.append({
                        "txn_id": random_id("txn"),
                        "timestamp": ts,
                        "src_account": src.account_id,
                        "dst_account": dst.account_id,
                        "amount": amt,
                        "src_country": src.country,
                        "dst_country": dst.country,
                        "currency": CURRENCIES.get(src.country, "EUR"),
                        "channel": "p2p",
                        "is_cross_border": int(src.country != dst.country),
                        "is_off_hours": 1,
                        "is_planted_mule_edge": 1,
                        "planted_cluster_id": cluster_id,
                    })
                    planted_edges.append((src.account_id, dst.account_id, cluster_id, "chain"))

    txns_df = pd.DataFrame(rows)
    planted_df = pd.DataFrame(planted_edges, columns=["src_account", "dst_account", "cluster_id", "edge_type"])
    return txns_df, planted_df


# ----------------------------
# Feature enrichment
# ----------------------------

def enrich_transactions(transactions: pd.DataFrame, accounts_df: pd.DataFrame) -> pd.DataFrame:
    acct_meta = accounts_df.set_index("account_id")

    def lookup(account_id: str, col: str):
        return acct_meta.loc[account_id, col]

    transactions = transactions.copy()
    transactions["src_kyc_risk"] = transactions["src_account"].map(lambda x: lookup(x, "kyc_risk"))
    transactions["dst_kyc_risk"] = transactions["dst_account"].map(lambda x: lookup(x, "kyc_risk"))
    transactions["src_age_days"] = transactions["src_account"].map(lambda x: lookup(x, "age_days"))
    transactions["dst_age_days"] = transactions["dst_account"].map(lambda x: lookup(x, "age_days"))
    transactions["shared_device"] = transactions.apply(
        lambda r: int(lookup(r["src_account"], "device_id") == lookup(r["dst_account"], "device_id")),
        axis=1
    )
    transactions["shared_ip"] = transactions.apply(
        lambda r: int(lookup(r["src_account"], "ip_id") == lookup(r["dst_account"], "ip_id")),
        axis=1
    )
    transactions["near_threshold"] = transactions["amount"].between(90, 120).astype(int)
    transactions["src_new_account"] = (transactions["src_age_days"] < 45).astype(int)
    transactions["dst_new_account"] = (transactions["dst_age_days"] < 45).astype(int)
    return transactions


def build_account_labels(accounts: Dict[str, Account]) -> pd.DataFrame:
    return pd.DataFrame([
        {
            "account_id": a.account_id,
            "is_mule": int(a.segment == "mule"),
            "mule_role": a.mule_role,
            "planted_cluster_id": a.cluster_id,
        }
        for a in accounts.values()
    ])


# ----------------------------
# Main
# ----------------------------

def main() -> None:
    start_date = datetime(2025, 1, 1)
    end_date = datetime(2025, 3, 31)

    print("Generating accounts...")
    accounts_df, accounts, mule_clusters = generate_accounts(
        n_accounts=N_ACCOUNTS,
        n_mule_clusters=N_MULE_CLUSTERS,
        start_date=start_date,
        end_date=end_date,
    )

    print("Generating normal transactions...")
    normal_txns = generate_normal_transactions(
        accounts=accounts,
        n_txns=N_NORMAL_TXNS,
        start_date=start_date,
        end_date=end_date,
    )

    print("Generating planted mule transactions...")
    mule_txns, mule_edges_df = generate_mule_transactions(
        accounts=accounts,
        mule_clusters=mule_clusters,
        start_date=start_date,
        end_date=end_date,
    )

    print("Combining and enriching...")
    transactions = pd.concat([normal_txns, mule_txns], ignore_index=True)
    transactions = transactions.sort_values("timestamp").reset_index(drop=True)
    transactions = enrich_transactions(transactions, accounts_df)

    account_labels = build_account_labels(accounts)

    print("Writing CSV files...")
    accounts_df.to_csv("accounts.csv", index=False)
    transactions.to_csv("transactions.csv", index=False)
    account_labels.to_csv("account_labels.csv", index=False)
    mule_edges_df.to_csv("mule_edges_groundtruth.csv", index=False)

    # quick summary
    n_mules = int(account_labels["is_mule"].sum())
    frac_mules = n_mules / len(account_labels)

    print("\nDone.")
    print(f"Accounts: {len(accounts_df):,}")
    print(f"Transactions: {len(transactions):,}")
    print(f"Mule accounts: {n_mules:,} ({frac_mules:.2%})")
    print(f"Planted mule edges: {len(mule_edges_df):,}")

    suspicious_txn_rate = transactions["is_planted_mule_edge"].mean()
    print(f"Planted suspicious transaction share: {suspicious_txn_rate:.2%}")

    print("\nFiles created:")
    print(" - accounts.csv")
    print(" - transactions.csv")
    print(" - account_labels.csv")
    print(" - mule_edges_groundtruth.csv")


if __name__ == "__main__":
    main()

Generating accounts...
Generating normal transactions...
Generating planted mule transactions...
Combining and enriching...
Writing CSV files...

Done.
Accounts: 2,500
Transactions: 24,622
Mule accounts: 116 (4.64%)
Planted mule edges: 2,622
Planted suspicious transaction share: 10.65%

Files created:
 - accounts.csv
 - transactions.csv
 - account_labels.csv
 - mule_edges_groundtruth.csv
